In [4]:
!pip install --upgrade google-colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.0 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [6]:
import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4


In [7]:
import os
import zipfile
import random
import shutil
import pandas as pd
import numpy as np

from tqdm import tqdm

In [8]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

DATASET_PATH = os.path.join(PROJECT_PATH, "dataset")

print(PROJECT_PATH)
print(DATASET_PATH)

/content/drive/MyDrive/Deepfake_Project
/content/drive/MyDrive/Deepfake_Project/dataset


In [9]:
folders = [
    "dataset",
    "preprocessing",
    "extracted_frames",
    "sampled_frames",
    "faces",
    "aligned_faces",
    "enhanced_faces",
    "spatial_features",
    "temporal_features",
    "models",
    "checkpoints",
    "results"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print("✅ Project folders created successfully.")

✅ Project folders created successfully.


In [10]:
print(os.listdir(PROJECT_PATH))

['dataset', 'preprocessing', 'extracted_frames', 'sampled_frames', 'faces', 'aligned_faces', 'enhanced_faces', 'spatial_features', 'temporal_features', 'models', 'checkpoints', 'results']


In [11]:
zip_path = os.path.join(DATASET_PATH, "ff_subset_500_500.zip")

print("ZIP Exists:", os.path.exists(zip_path))

ZIP Exists: True


In [12]:
extract_path = DATASET_PATH

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully!")

✅ Dataset extracted successfully!


In [13]:
print(os.listdir(DATASET_PATH))

['ff_subset_500_500.zip', 'Deepfakes', 'original']


In [15]:
real_path = os.path.join(DATASET_PATH, "original")
fake_path = os.path.join(DATASET_PATH, "Deepfakes")

print(os.listdir(real_path)[:5])
print(os.listdir(fake_path)[:5])

['041.mp4', '928.mp4', '529.mp4', '976.mp4', '678.mp4']
['482_465.mp4', '103_082.mp4', '234_187.mp4', '830_828.mp4', '398_457.mp4']


In [16]:
print("Real Videos :", len(os.listdir(real_path)))
print("Fake Videos :", len(os.listdir(fake_path)))

Real Videos : 500
Fake Videos : 500


In [17]:
import random

random.seed(42)

real_files = sorted(os.listdir(real_path))
fake_files = sorted(os.listdir(fake_path))

random.shuffle(real_files)
random.shuffle(fake_files)

train_real = real_files[:300]
val_real   = real_files[300:400]
test_real  = real_files[400:500]

train_fake = fake_files[:300]
val_fake   = fake_files[300:400]
test_fake  = fake_files[400:500]

In [18]:
import pandas as pd

def create_dataframe(files, folder, label):
    return pd.DataFrame({
        "filename": files,
        "folder": folder,
        "label": label
    })

train_df = pd.concat([
    create_dataframe(train_real, "original", 0),
    create_dataframe(train_fake, "Deepfakes", 1)
], ignore_index=True)

val_df = pd.concat([
    create_dataframe(val_real, "original", 0),
    create_dataframe(val_fake, "Deepfakes", 1)
], ignore_index=True)

test_df = pd.concat([
    create_dataframe(test_real, "original", 0),
    create_dataframe(test_fake, "Deepfakes", 1)
], ignore_index=True)

# Shuffle
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [19]:
train_df.to_csv(os.path.join(DATASET_PATH, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATASET_PATH, "val.csv"), index=False)
test_df.to_csv(os.path.join(DATASET_PATH, "test.csv"), index=False)

print("✅ CSV files created successfully.")

✅ CSV files created successfully.


In [20]:
print("Train :", len(train_df))
print("Validation :", len(val_df))
print("Test :", len(test_df))

print("\nTrain Labels")
print(train_df["label"].value_counts())

print("\nValidation Labels")
print(val_df["label"].value_counts())

print("\nTest Labels")
print(test_df["label"].value_counts())

Train : 600
Validation : 200
Test : 200

Train Labels
label
0    300
1    300
Name: count, dtype: int64

Validation Labels
label
0    100
1    100
Name: count, dtype: int64

Test Labels
label
0    100
1    100
Name: count, dtype: int64


In [21]:
print(train_df.head())

      filename     folder  label
0      078.mp4   original      0
1  082_103.mp4  Deepfakes      1
2  490_534.mp4  Deepfakes      1
3      642.mp4   original      0
4      327.mp4   original      0
